# Generative adversarial networks (GANs)

A generative adversarial network (GAN) is a non-cooperative zero-sum game where two networks compete against each other.

A GAN consists of two competing neural networks:

- Generator ($G$): Takes random noise as input and generates artificial data, such as an image
- Discriminator ($D$): Takes an image as input and predicts the probability that it came from the real dataset rather than the Generator.

The objective function is

$$
\min_G \max_D V(D,G)
=
\mathbb{E}_{x \sim p_{\text{data}}(x)}[\log D(x)]
+
\mathbb{E}_{z \sim p_z(z)}[\log(1-D(G(z)))]
$$

- $\min_G$: The Generator tries to minimise the objective function
- $\max_D$: The Discriminator tries to maximise the objective function.

The first term, $\mathbb{E}_{x \sim p_{\text{data}}(x)}[\log D(x)]$, refers to real data:

- $x$: A real image from the training dataset.
- $p_{\text{data}}(x)$: The probability distribution of real training images.
- $D(x)$: The Discriminator's predicted probability that the image is real.
- $\log D(x)$: The logarithm of that probability.
- $\mathbb{E}$: The expected value (average) over samples from the real data distribution.
- The Discriminator wants $D(x)$ to be close to 1 for real images

The second term, $\mathbb{E}_{z \sim p_z(z)}[\log(1-D(G(z)))]$, refers to generated data:

- $z$: A random noise vector sampled from a probability distribution.
- $p_z(z)$: The probability distribution from which random noise is sampled.
- $G(z)$: The Generator's output, such as a fake image.
- $D(G(z))$: The Discriminator's predicted probability that the generated image is real.
- $\log(1-D(G(z)))$: The logarithm of the probability that the Discriminator identifies a generated image as fake.
- The Discriminator wants $D(G(z))$ to be close to 0 for generated images.

A GAN is optimal when the Generator learns the real data distribution:
$$
p_g(x)=p_{\text{data}}(x)
$$

where $p_g(x)$ is the probability distribution of generated data.

In this situation, the Discriminator cannot distinguish real and generated images.

Its optimal prediction becomes:
$$
D^*(x)=\frac{p_{\text{data}}(x)}
{p_{\text{data}}(x)+p_g(x)}
=0.5
$$

- it assigns an equal probability of an image being real or fake.

The general way in which a GAN works is:

- The Generator creates a fake image.
- The Discriminator examines it and decides whether it is real or fake.
- The Generator learns from its mistakes and tries to create more convincing images.
- The Discriminator also improves at detecting fake images.
- They repeat this process, continually improving.

The data being generated may not necessarily be images. 

## Properties of GANs

GANs can generate very realistic data, but they are notoriously difficult to train. There are four main problems.

### Non-convergence

The Generator and Discriminator can continuously chase each other without reaching a stable equilibrium.

- The Generator improves at creating realistic images.
- The Discriminator improves at identifying fake images.
- The Generator then has to adapt again to the improved Discriminator.
- This cycle can continue without either network settling.

Essentially, the two networks keep trying to outsmart each other, making it difficult for training to stabilise.

### Diminishing gradients

The Generator can receive very small gradients, making it difficult to learn.

- Initially, the Generator produces poor-quality images.
- The Discriminator easily identifies them as fake, outputting a probability close to 0.
- The Generator receives little useful learning signal from the original minmax objective when the Discriminator becomes too good.
- Its weights therefore update very slowly.

The Discriminator becomes so good at detecting fake images that the Generator struggles to learn how to improve.

### Difficult to balance

The Generator and Discriminator need to improve at roughly compatible rates.

There are two possible situations:

- Discriminator too strong: It easily identifies fake images, potentially leaving the Generator with weak gradients.
- Generator too strong: The Discriminator cannot reliably distinguish real from fake, making its feedback less useful.

Ideally, neither network should become overwhelmingly better than the other.

If one network becomes much better than the other, the training process becomes less effective.

### Mode collapse

The Generator learns to produce only a limited variety of outputs, rather than representing the full diversity of the training data.

For example, imagine training a GAN on handwritten digits (0-9).

Instead of learning to generate all ten digits, the Generator might discover that producing convincing 7s consistently fools the Discriminator.

It then produces mostly 7s, rather than generating a diverse range of digits.

In summary, the Generator finds a small number of outputs that fool the Discriminator and keeps producing them instead of learning to generate diverse data.

## Lipschitz function

A Lipschitz function is a function whose output cannot change too rapidly relative to changes in its input.

A function $f$ is called K-Lipschitz continuous if:
$$
|f(x_1)-f(x_2)| \leq K|x_1-x_2|
$$

where:
- $x_1,x_2$: Two input values.
- $f(x_1),f(x_2)$: Their corresponding outputs.
- $K$: A constant that limits how quickly the function can change.

In simple terms, if the inputs are close together, their outputs cannot be arbitrarily far apart.

## Wasserstein GANs

A standard GAN uses a Discriminator to classify images as real or fake.

A Wasserstein GAN (WGAN) uses a slightly different approach: it replaces the Discriminator with a Critic.

The Critic does not classify images as real or fake. Instead, it assigns scores to images:

- Real images should receive higher scores.
- Generated images should receive lower scores.

The aim is to measure how different the generated data distribution is from the real data distribution.

## Conditional GANs

A Conditional GAN (cGAN) is a GAN where the Generator and Discriminator are both given additional information (called a condition) to influence what they generate and evaluate.

- In a standard GAN, the Generator receives random noise and generates an image without being told what the image should contain.
- In a conditional GAN, the Generator also receives a label, allowing us to control what it generates.

The important part of a conditional GAN is that the label information is fed into both the Generator and Discriminator.

- Generator (G):
  - Receives random noise $z$ and label $y$.
  - Uses the label to determine what type of image to generate.
- Discriminator (D):
  - Receives an image $x$ and label $y$.
  - Determines whether the image is real or fake, given the supplied label.

This means the Discriminator checks not only whether the image looks realistic, but also whether it matches the supplied label.

## Information maximising GANs

An Information Maximising GAN (InfoGAN) is a type of GAN that learns to generate data with controllable features by discovering meaningful patterns in its random input.

The key idea is that InfoGAN encourages the Generator to learn which parts of its input control particular features of the generated output.

For example, when generating handwritten digits, it might learn that:

- One latent variable controls the digit's rotation.
- Another controls its thickness.
- Another controls its writing style.

Importantly, it learns these relationships without needing labelled training data (unsupervised learning).

Changing one latent variable might rotate a digit while keeping its identity and other characteristics relatively unchanged.

InfoGAN divides the Generator's input into two parts:

1. Random noise ($z$): Captures general variation in the generated data.
2. Latent code ($c$): Contains variables that InfoGAN tries to associate with meaningful, interpretable features.
The Generator receives both:
$$
x=G(z,c)
$$

For example:
- $z$ provides general randomness.
- $c_1$ might control rotation.
- $c_2$ might control thickness.
- $c_3$ might control digit identity.
- 
The important distinction is that InfoGAN does not explicitly receive labels telling it what these variables should represent. It attempts to discover useful relationships automatically.

InfoGAN maximises the mutual information between the latent code $c$ and the generated observation $x$.

- Mutual information measures how much information one variable provides about another.
- The goal is to make the generated output informative about the latent code.
- By maximising mutual information, InfoGAN encourages the Generator to make the latent codes meaningful and recoverable from its output.